In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# קלטים ופלטים של Estimator

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

דף זה נותן סקירה של הקלטים והפלטים של ה-primitive של Qiskit Runtime Estimator, שמריץ עומסי עבודה על משאבי מחשוב IBM Quantum&reg;. Estimator מאפשר לך להגדיר עומסי עבודה מוּוקטרים בצורה יעילה באמצעות מבנה נתונים הנקרא [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). הם משמשים כקלטים למתודת [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) עבור ה-primitive של Estimator, שמריצה את עומס העבודה המוגדר כעבודה. לאחר השלמת העבודה, התוצאות מוחזרות בפורמט התלוי גם ב-PUBs שנעשה בהם שימוש וגם באפשרויות ה-runtime שצוינו מה-primitive.

## קלטים
כל PUB נמצא בפורמט זה:

(`<מעגל בודד>`, `<observable אחד או יותר>`, `<ערכי פרמטרים אופציונליים>`, `<דיוק אופציונלי>`),

ה-`parameter values` האופציונליים יכולים להיות רשימה או פרמטר בודד. אלמנטים מה-observables ומערכי הפרמטרים משולבים בעקבות כללי שידור NumPy כמתואר בנושא [קלטים ופלטים של Primitive](primitive-input-output#broadcasting-rules), ואומדן ערך ציפייה אחד מוחזר לכל אלמנט בצורה המשודרת.

> **Note:** אם הקלט מכיל מדידות, הן מתעלמות.

עבור ה-primitive של Estimator, PUB יכול להכיל לכל היותר ארבעה ערכים:
- `QuantumCircuit` בודד, שעשוי להכיל אובייקט [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) אחד או יותר
- רשימה של observable אחד או יותר, שמציינים את ערכי הציפייה לאמוד, מסודרים במערך (לדוגמה, observable בודד המיוצג כמערך 0-d, רשימת observables כמערך 1-d, וכן הלאה). הנתונים יכולים להיות בכל פורמט `ObservablesArrayLike` כמו `Pauli`, `SparsePauliOp`, `PauliList` או `str`.
   > **Note:** - Observables מתחלפים **באותו PUB** מקובצים יחד באמצעות [שיטה זו](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Observables מתחלפים ב-PUBs שונים, גם אם יש להם אותו מעגל, אינם מוערכים באמצעות אותה מדידה. כל PUB מייצג בסיס שונה למדידה, ולכן נדרשות מדידות נפרדות לכל PUB.
>    - כדי להבטיח ש-observables מתחלפים מוערכים באמצעות אותה מדידה, קבץ אותם בתוך אותו PUB.
- אוסף ערכי פרמטרים לאגד עם המעגל. זה יכול להיות מוגדר כאובייקט בודד דמוי-מערך שבו המדד האחרון הוא מעל אובייקטי `Parameter` של המעגל, או שמוּשמט (או שווה ערך, מוגדר ל-`None`) אם למעגל אין אובייקטי `Parameter`.
- (אופציונלי) דיוק יעד לאמידת ערכי ציפייה
---

הקוד הבא מדגים סט לדוגמה של קלטים מוּוקטרים ל-primitive של `Estimator` ומריץ אותם על Backend של IBM&reg; כאובייקט `RuntimeJobV2` בודד.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## פלטים
לאחר שנשלח PUB אחד או יותר ל-QPU להרצה ועבודה מושלמת בהצלחה, הנתונים מוחזרים כאובייקט מכיל [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) שאליו ניגשים על ידי קריאה למתודת `RuntimeJobV2.result()`.

ה-`PrimitiveResult` מכיל רשימה איטרבילית של אובייקטי [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) שמכילים את תוצאות ההרצה לכל PUB.

כל אלמנט ברשימה זו מתאים לכל PUB שנשלח למתודת `run()` של ה-primitive (לדוגמה, עבודה שנשלחת עם 20 PUBs תחזיר אובייקט `PrimitiveResult` שמכיל רשימה של 20 אובייקטי `PubResult`, אחד המתאים לכל PUB).

כל `PubResult` עבור ה-primitive של Estimator מכיל לפחות מערך ערכי ציפייה (`PubResult.data.evs`) וסטיות תקן משויכות (כ-`PubResult.data.stds` או `PubResult.data.ensemble_standard_error` בהתאם ל-`resilience_level` שנעשה בו שימוש), אך יכול להכיל נתונים נוספים בהתאם לאפשרויות הפחתת השגיאות שצוינו.

כל אובייקט `PubResult` מחזיק גם מאפיין `data` וגם מאפיין `metadata`.
    - מאפיין ה-`data` הוא [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) מותאם אישית שמכיל את ערכי המדידה בפועל, סטיות תקן, וכן הלאה.
    - ל-`DataBin` יש מאפיינים שונים בהתאם לצורה או מבנה ה-PUB המשויך וכן לאפשרויות הפחתת השגיאות שצוינו על ידי ה-primitive שנעשה בו שימוש לשליחת העבודה (לדוגמה, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) או [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - מאפיין ה-`metadata` מכיל מידע על ה-runtime ואפשרויות הפחתת השגיאות שנעשה בהם שימוש (מוסבר בהמשך בחלק [מטא-נתוני תוצאה](#result-metadata) בדף זה).

להלן מתאר חזותי של מבנה הנתונים `PrimitiveResult` עבור פלט Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

בפשטות, עבודה בודדת מחזירה אובייקט `PrimitiveResult` ומכילה רשימה של אובייקט `PubResult` אחד או יותר. אובייקטי ה-`PubResult` הללו אחסנים את נתוני המדידה לכל PUB שנשלח לעבודה.

קטע הקוד הבא מתאר את הפורמט של `PrimitiveResult` (ושל `PubResult` המשויך) עבור העבודה שנוצרה לעיל.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
